# `country_risk_premium(country=None, on=None, revision=None)`

The extra equity premium Damodaran adds for investing in each country, from `macro.damodaran_country_risk`. Two estimates per row: `crp` from the country's rating (its default spread times the edition's volatility multiplier; the US's is its raw spread), and `cds_crp` from the CDS market (NULL where no CDS trades).

Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

`country` is an ISO alpha-3 code (`AE-AZ`, `AE-RK`, `AE-SH` for the three emirates) or a name from that edition, in any case and spacing; `None` gives every country. `on` picks the edition in force on that date (default: today in US/Eastern), and `revision` defaults to its newest. Values are Damodaran's, as published and never recomputed, in fractions: 0.0442 is 4.42%.

In [1]:
from datetime import date

import polars as pl
from IPython.display import Markdown

from arora_data.country_risk_premium import CountryRiskPremium
from arora_data.sync import country_risk_premium, damodaran_editions, sovereign_spread
from arora_data.dates import today_et
from arora_data.frames import to_df


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## One country, by code or name
The same row either way, in any case and spacing.

In [2]:
assert country_risk_premium("TUR") == country_risk_premium("turkey") == country_risk_premium(" TURKEY ")
show(country_risk_premium("TUR"))

1 rows

edition_date|revision|entity_code|published_name|entity_type|region|crp|cds_crp
---|---|---|---|---|---|---|---
2026-07-01|1|TUR|Turkey|country|Western Europe|0.0445120778|0.0419713792

## The whole edition, highest premium first
In percent for reading; the values themselves stay fractions.

In [3]:
crp = to_df(country_risk_premium(), CountryRiskPremium)
print(crp.height, "rows, edition", crp["edition_date"][0])
crp.sort("crp", descending=True).head(10).select("entity_code", "published_name", (pl.col("crp") * 100).alias("crp, %"), (pl.col("cds_crp") * 100).alias("cds_crp, %"))

178 rows, edition 2026-07-01


entity_code,published_name,"crp, %","cds_crp, %"
str,str,"decimal[38,10]","decimal[38,10]"
"""BLR""","""Belarus""",27.2036717100,null
"""BRB""","""Barbados""",27.2036717100,null
"""LBN""","""Lebanon""",27.2036717100,null
"""SDN""","""Sudan""",27.2036717100,null
"""CUB""","""Cuba""",14.8519057100,null
"""MMR""","""Myanmar""",14.8519057100,null
"""MWI""","""Malawi""",14.8519057100,null
"""PRK""","""Korea, D.P.R.""",14.8519057100,null
"""UKR""","""Ukraine""",14.8519057100,null


## How crp is built
The default spread times the edition's multiplier, except the US, whose premium is its raw Aa1 spread.

In [4]:
mult = [e for e in damodaran_editions() if e.latest][-1].vol_multiplier
for c in ["ARE", "BRA", "USA"]:
    p, s = country_risk_premium(c)[0], sovereign_spread(c)[0]
    print(f"{c}: crp {p.crp}  spread x multiplier {s.default_spread * mult:.10f}  raw spread {s.default_spread}")

ARE: crp 0.0061095009  spread x multiplier 0.0061095009  raw spread 0.0039302145


BRA: crp 0.0309838973  spread x multiplier 0.0309838974  raw spread 0.0199318022


USA: crp 0.0021863537  spread x multiplier 0.0033986770  raw spread 0.0021863537


## The rating-based and CDS-based estimates side by side
Only countries with a traded CDS have both. The biggest gaps between the two:

In [5]:
(crp.drop_nulls("cds_crp").with_columns((pl.col("cds_crp") - pl.col("crp")).alias("cds minus rating"))
 .sort("cds minus rating").select("entity_code", "published_name", "crp", "cds_crp", "cds minus rating"))

entity_code,published_name,crp,cds_crp,cds minus rating
str,str,"decimal[38,10]","decimal[38,10]","decimal[38,10]"
"""ZMB""","""Zambia""",0.1114256589,0.0450803703,-0.0663452886
"""EGY""","""Egypt""",0.0928062276,0.0531637470,-0.0396424806
"""IRQ""","""Iraq""",0.0928062276,0.0579826831,-0.0348235445
"""NGA""","""Nigeria""",0.0804417616,0.0506765542,-0.0297652074
"""SLV""","""El Salvador""",0.0804417616,0.0522310497,-0.0282107119
…,…,…,…,…
"""COL""","""Colombia""",0.0272018253,0.0369969935,0.0097951682
"""NIC""","""Nicaragua""",0.0680772955,0.0803674187,0.0122901232
"""TUN""","""Tunisia""",0.0928062276,0.1078819895,0.0150757619


## As of a date, and one revision
Any date in March 2026 gets the January edition at its newest revision (4, the Feb 16 re-issue); `revision=1` gets it as first posted.

In [6]:
for kwargs in [dict(on=date(2026, 3, 31)), dict(on=date(2026, 3, 31), revision=1)]:
    r = country_risk_premium("KEN", **kwargs)[0]
    print(kwargs, r.edition_date, "revision", r.revision, "crp", r.crp)

{'on': datetime.date(2026, 3, 31)} 2026-01-01 revision 4 crp 0.0841434947


{'on': datetime.date(2026, 3, 31), 'revision': 1} 2026-01-01 revision 1 crp 0.0970769432


## One country across editions
Turkey's premium in every edition, each at its newest revision.

In [7]:
for e in damodaran_editions():
    if e.latest:
        r = country_risk_premium("TUR", on=e.edition_date)[0]
        print(r.edition_date, f"{r.crp * 100:.2f}%")

2015-01-01 3.30%


2016-07-01 3.46%


2017-01-05 3.55%


2018-01-01 2.88%


2019-01-01 5.00%


2020-01-01 4.44%


... (5 outputs left out)

2025-07-01 6.66%


2026-01-01 4.66%


2026-04-01 5.25%


2026-07-01 4.45%


## Rows FMP never carried
The three emirates he lists apart from the UAE, and St. Vincent & the Grenadines.

In [8]:
show([r for r in country_risk_premium() if r.entity_code in ("AE-AZ", "AE-RK", "AE-SH", "ARE", "VCT")])

5 rows

edition_date|revision|entity_code|published_name|entity_type|region|crp|cds_crp
---|---|---|---|---|---|---|---
2026-07-01|1|AE-AZ|Abu Dhabi|subnational|Middle East|0.0061095009|0.0080833767
2026-07-01|1|AE-RK|Ras Al Khaimah (Emirate of)|subnational|Middle East|0.0148373593|None
2026-07-01|1|AE-SH|Sharjah|subnational|Middle East|0.0309838973|None
2026-07-01|1|ARE|United Arab Emirates|country|Middle East|0.0061095009|None
2026-07-01|1|VCT|St. Vincent & the Grenadines|country|Caribbean|0.0804417616|None

## Known source issue
In the 2026-07-01 edition Barbados's Moody's rating is typed as C in his file (B2 three months earlier), so it carries Belarus's values. Stored as he published it.

In [9]:
show(country_risk_premium("BRB", on=date(2026, 4, 1)) + country_risk_premium("BRB", on=date(2026, 7, 1)) + country_risk_premium("BLR", on=date(2026, 7, 1)))

3 rows

edition_date|revision|entity_code|published_name|entity_type|region|crp|cds_crp
---|---|---|---|---|---|---|---
2026-04-01|1|BRB|Barbados|country|Caribbean|0.0803143957|None
2026-07-01|1|BRB|Barbados|country|Caribbean|0.2720367171|None
2026-07-01|1|BLR|Belarus|country|Eastern Europe & Russia|0.2720367171|None

## Bad input
An unknown country, a date before the first edition (2015-01-01), or a revision the edition doesn't have raises a clear error.

In [10]:
for kwargs in [dict(country="South Korea"), dict(country="US"), dict(on=date(2014, 12, 31)),
               dict(on=date(2026, 3, 31), revision=7)]:
    try:
        country_risk_premium(**kwargs)
    except ValueError as e:
        print(e)

'South Korea' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


'US' is not a stored country in Damodaran's 2026-07-01 edition: use an ISO alpha-3 code or a name from that edition, such as USA or 'United States'


no Damodaran edition on or before 2014-12-31: the first is 2015-01-01


Damodaran's 2026-01-01 edition has no revision 7: it has 1, 2, 3, 4
